# FINAL Export — CF-XHPO Revision Consolidation

Run this **last**, from `hpo_pdt_dataset/` (same place as the other notebooks).
It performs the closing computations itself, so **no second pass of
`revision_A_reanalysis_v2` is needed**:

1. Merges each dataset's confirmation CSV into the generator-vs-retrieval
   comparison (the S9 step) — including MVTec once its Section R has run.
2. Patches `end_to_end_latency_*` / `energy_revised_*` with the measured parser
   latencies from `parser_eval_latency.json` (replacing the NaN placeholders),
   recovering the retraining time exactly from the stored speed-up.
3. Runs a completeness gate over every expected artifact.
4. Writes **7 final files** into `final_results/` — the complete, structured
   record of the revision, sized for upload and analysis:

| # | File | Contents |
|---|---|---|
| 1 | `final_manifest.json` | file inventory, hashes, timestamps, env versions, completeness |
| 2 | `final_core_tables_mvtec.json` | all MVTec tables (S1–S11) |
| 3 | `final_core_tables_proditec.json` | all Proditec tables (S1–S11) |
| 4 | `final_parser_eval.json` | v2 + as-deployed summaries, per-utterance, semantic diagnostic, latency |
| 5 | `final_user_study.json` | valid-only Table 9, what-if, attempts, violations, task-level export |
| 6 | `final_query_suite_detail.json` | full per-cell query-suite results, both datasets |
| 7 | `final_report.txt` | human-readable completeness + sanity report |

In [1]:
# ============================================================
# F0) Setup: locate roots, output dir, helpers
# ============================================================
from pathlib import Path
import json, hashlib, datetime
import numpy as np
import pandas as pd

CWD = Path.cwd().resolve()
BASE = CWD.parent

def find_roots():
    roots = {}
    for ro in list(BASE.rglob("revision_outputs")):
        par = ro.parent; low = par.name.lower()
        tag = ("mvtec" if "mvtec" in low else
               "proditec" if ("pdt" in low or "proditec" in low) else None)
        if tag and tag not in roots:
            roots[tag] = par
    return roots

ROOTS = find_roots()
assert ROOTS, f"No revision_outputs trees found under {BASE}"
print("Roots:", {k: str(v) for k, v in ROOTS.items()})

FINAL = CWD / "final_results"
FINAL.mkdir(exist_ok=True)

def T(ds):  return ROOTS[ds] / "revision_outputs" / "tables"
def RV(ds): return ROOTS[ds] / "revision_outputs"

def read_csv(p):
    return pd.read_csv(p) if Path(p).exists() else None

def stat(p):
    p = Path(p); st = p.stat()
    return dict(path=str(p), size=st.st_size,
                modified=datetime.datetime.fromtimestamp(st.st_mtime)
                         .isoformat(timespec="seconds"),
                md5=hashlib.md5(p.read_bytes()).hexdigest()[:12])

def recs(df):
    return json.loads(df.to_json(orient="records")) if df is not None else None


Roots: {'mvtec': '/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class', 'proditec': '/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_pdt_dataset'}


In [2]:
# ============================================================
# F1) S9-equivalent merge: generator vs retrieval, CONFIRMED
#     (writes the canonical CSV back into tables/ for each ds)
# ============================================================
for ds in ROOTS:
    cand = read_csv(T(ds) / f"baseline_confirm_candidates_{ds}.csv")
    conf = read_csv(T(ds) / f"baseline_confirmation_{ds}.csv")
    if conf is None or not len(conf):
        print(f"[{ds}] confirmation CSV missing/empty — Section R of notebook "
              f"5{'th' if ds=='mvtec' else 'b'} has not produced results yet.")
        continue
    ok = conf[conf.get("status", "ok") == "ok"] if "status" in conf.columns else conf
    if "gap_test_minus_surrogate_pred" not in ok.columns and        {"f1_test", "surrogate_pred"} <= set(ok.columns):
        ok = ok.assign(gap_test_minus_surrogate_pred=
                       ok["f1_test"] - ok["surrogate_pred"])
    summ = (ok.groupby(["qid", "method"])
              .agg(n=("method", "size"),
                   surrogate_f1=("surrogate_pred", "mean"),
                   confirmed_f1=("f1_test", "mean"),
                   gap=("gap_test_minus_surrogate_pred", "mean"),
                   runtime_sec=("runtime_sec", "mean"))
              .reset_index().round(4))
    summ.to_csv(T(ds) / f"baseline_vs_generator_confirmed_{ds}.csv", index=False)
    print(f"[{ds}] confirmed comparison written "
          f"({len(ok)} ok rows, {int((conf.get('status','ok')!='ok').sum()) if 'status' in conf.columns else 0} errors):")
    display(summ)


[mvtec] confirmed comparison written (6 ok rows, 0 errors):


,qid,method,n,surrogate_f1,confirmed_f1,gap,runtime_sec
0,tight_band_q,DiCE,2,0.8015,0.7930,-0.0085,923.5734
1,tight_band_q,LogTopK,2,0.8083,0.7935,-0.0148,497.4468
2,tight_band_q,NearestFeasible,2,0.8001,0.8006,0.0005,344.1576


[proditec] confirmed comparison written (6 ok rows, 0 errors):


,qid,method,n,surrogate_f1,confirmed_f1,gap,runtime_sec
0,tight_band,DiCE,2,0.8962,0.8760,-0.0202,72.2677
1,tight_band,LogTopK,2,0.8976,0.8878,-0.0098,25.8614
2,tight_band,NearestFeasible,2,0.8914,0.8822,-0.0092,28.2409


In [3]:
# ============================================================
# F2) Patch latency + energy with measured parser figures
# ============================================================
PATCH_W, SUR_W, GRID = 300.0, 65.0, 0.66

lat_json = None
for ds in ROOTS:
    p = RV(ds) / "parser_eval_latency.json"
    if p.exists():
        cand = json.load(open(p))
        if lat_json is None or p.stat().st_mtime > lat_json["_mtime"]:
            lat_json = dict(cand, _mtime=p.stat().st_mtime, _src=str(p))
assert lat_json, "parser_eval_latency.json not found in any revision_outputs/"
parse_s = float(lat_json["parse_mean_s"])
explain_s = float(lat_json.get("explain_mean_s") or np.nan)
print(f"Parser figures from {lat_json['_src']}: parse={parse_s:.1f}s, "
      f"explain={explain_s:.1f}s, dtype={lat_json.get('model_dtype')}")

for ds in ROOTS:
    lat = read_csv(T(ds) / f"end_to_end_latency_{ds}.csv")
    if lat is None:
        print(f"[{ds}] end_to_end_latency missing — run revision_A S11 first.")
        continue
    e2e_old = lat["end_to_end_s"].astype(float)
    retrain_s = float((lat["speedup_vs_retraining"] * e2e_old).median())
    lat["parse_s"] = parse_s
    lat["explain_s"] = explain_s
    lat["end_to_end_s"] = lat[["parse_s", "anchor_scoring_s", "generation_s",
                               "rank_s", "explain_s"]].sum(axis=1, skipna=True)
    lat["speedup_vs_retraining"] = retrain_s / lat["end_to_end_s"]
    lat = lat.round(4)
    lat.to_csv(T(ds) / f"end_to_end_latency_{ds}.csv", index=False)

    cpu_s = lat[["anchor_scoring_s", "generation_s", "rank_s"]].sum(axis=1)
    gpu_s = lat[["parse_s", "explain_s"]].sum(axis=1)
    en = pd.DataFrame({
        "method": lat["method"],
        "energy_per_query_Wh": (cpu_s * SUR_W + gpu_s * PATCH_W) / 3600,
        "retrain_energy_per_query_Wh": retrain_s * PATCH_W / 3600,
    })
    en["avoided_per_1000_queries_kWh"] = (
        en["retrain_energy_per_query_Wh"] - en["energy_per_query_Wh"])
    en["co2_avoided_per_1000_queries_kg"] = en["avoided_per_1000_queries_kWh"] * GRID
    en = en.round(4)
    en.to_csv(T(ds) / f"energy_revised_{ds}.csv", index=False)
    print(f"[{ds}] latency+energy patched (retrain={retrain_s:.1f}s):")
    display(lat[["method", "end_to_end_s", "speedup_vs_retraining"]])


Parser figures from /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_pdt_dataset/revision_outputs/parser_eval_latency.json: parse=56.9s, explain=84.5s, dtype=torch.float32
[mvtec] latency+energy patched (retrain=30.8s):


,method,end_to_end_s,speedup_vs_retraining
0,ACFX,241.2390,0.1275
1,CFNOW,146.5379,0.2099
2,DiCE,141.7664,0.2169
3,LogTopK,141.4035,0.2175
4,NearestFeasible,141.4039,0.2175


[proditec] latency+energy patched (retrain=47.4s):


,method,end_to_end_s,speedup_vs_retraining
0,ACFX,252.7564,0.1874
1,CFNOW,142.8822,0.3315
2,DiCE,141.7269,0.3342
3,LogTopK,141.3913,0.3350
4,NearestFeasible,141.3920,0.3350


In [4]:
# ============================================================
# F3) Completeness gate
# ============================================================
CORE = ["decomposition_per_method_{ds}.csv", "decomposition_per_candidate_{ds}.csv",
        "table4_variability_{ds}.csv",
        "gap_distance_correlations_descriptive_{ds}.csv",
        "soft_control_regression_{ds}.csv", "soft_control_spearman_{ds}.csv",
        "duplication_rate_{ds}.csv", "query_suite_{ds}.csv",
        "query_suite_aggregate_{ds}.csv", "baseline_confirm_candidates_{ds}.csv",
        "baseline_confirmation_{ds}.csv",
        "baseline_vs_generator_confirmed_{ds}.csv",
        "surrogate_split_check_{ds}.csv", "end_to_end_latency_{ds}.csv",
        "energy_revised_{ds}.csv"]
MVTEC_ONLY = ["two_mechanisms_summary_mvtec.csv",
              "two_mechanisms_per_candidate_mvtec.csv"]
SHARED = ["parser_eval_v2_summary.csv", "parser_eval_v2_per_utterance.csv",
          "parser_eval_v2_relative_diagnostic.csv", "parser_eval_v2_consistency.csv",
          "parser_eval_summary.csv", "parser_eval_latency.json",
          "table9_valid_only_split.csv", "whatif_validity_by_condition.csv",
          "interface_attempts_by_task.csv", "violations_breakdown.csv",
          "task_level_answers_export.csv"]

missing, empty = [], []
def check(p, label):
    p = Path(p)
    if not p.exists():
        missing.append(label); return
    if p.suffix == ".csv":
        try:
            if len(pd.read_csv(p)) == 0:
                empty.append(label)
        except Exception:
            empty.append(label + " (unreadable)")

for ds in ROOTS:
    for tmpl in CORE:
        check(T(ds) / tmpl.format(ds=ds), f"{ds}:{tmpl.format(ds=ds)}")
for tmpl in MVTEC_ONLY:
    if "mvtec" in ROOTS:
        check(T("mvtec") / tmpl, f"mvtec:{tmpl}")

def shared_find(name):
    hits = [RV(ds) / name for ds in ROOTS if (RV(ds) / name).exists()]
    return sorted(hits, key=lambda x: x.stat().st_mtime)[-1] if hits else None

shared_paths = {}
for name in SHARED:
    p = shared_find(name)
    if p is None:
        missing.append(f"shared:{name}")
    else:
        shared_paths[name] = p
        check(p, f"shared:{name}")

if missing or empty:
    print("!! INCOMPLETE — fix before export:")
    for m in missing: print("   MISSING:", m)
    for m in empty:   print("   EMPTY:  ", m)
else:
    print("OK — all expected artifacts present and non-empty.")
GATE_OK = not missing and not empty


OK — all expected artifacts present and non-empty.


In [5]:
# ============================================================
# F4) Write the 7 final files
# ============================================================
def load_all(ds, names):
    out = {}
    for tmpl in names:
        name = tmpl.format(ds=ds)
        df = read_csv(T(ds) / name)
        out[name.replace(".csv", "")] = recs(df)
    return out

CORE_NO_SUITE = [n for n in CORE if n != "query_suite_{ds}.csv"]
for ds in ROOTS:
    bundle = load_all(ds, CORE_NO_SUITE)
    if ds == "mvtec":
        bundle.update(load_all(ds, MVTEC_ONLY))
    bundle["_figures"] = sorted(p.name for p in
                                (RV(ds) / "figures").glob("*"))                          if (RV(ds) / "figures").is_dir() else []
    Path(FINAL / f"final_core_tables_{ds}.json").write_text(
        json.dumps(bundle, indent=1))
    print(f"final_core_tables_{ds}.json:",
          f"{(FINAL / f'final_core_tables_{ds}.json').stat().st_size/1024:.0f} KB")

suite = {ds: recs(read_csv(T(ds) / f"query_suite_{ds}.csv")) for ds in ROOTS}
Path(FINAL / "final_query_suite_detail.json").write_text(json.dumps(suite, indent=1))

pe = {}
for name in ["parser_eval_v2_summary.csv", "parser_eval_v2_per_utterance.csv",
             "parser_eval_v2_relative_diagnostic.csv",
             "parser_eval_v2_consistency.csv", "parser_eval_summary.csv"]:
    p = shared_paths.get(name)
    pe[name.replace(".csv", "")] = recs(read_csv(p)) if p else None
pe["parser_eval_latency"] = (json.load(open(shared_paths["parser_eval_latency.json"]))
                             if "parser_eval_latency.json" in shared_paths else None)
Path(FINAL / "final_parser_eval.json").write_text(json.dumps(pe, indent=1))

us = {}
for name in ["table9_valid_only_split.csv", "whatif_validity_by_condition.csv",
             "interface_attempts_by_task.csv", "violations_breakdown.csv",
             "task_level_answers_export.csv"]:
    p = shared_paths.get(name)
    us[name.replace(".csv", "")] = recs(read_csv(p)) if p else None
Path(FINAL / "final_user_study.json").write_text(json.dumps(us, indent=1))

try:
    from importlib.metadata import version
    env = {pkg: version(pkg) for pkg in
           ["catboost", "dice-ml", "torch", "transformers", "scikit-learn",
            "optuna", "jsonschema", "pandas", "numpy"]}
except Exception as e:
    env = {"error": repr(e)}

inventory = []
for ds in ROOTS:
    for p in sorted(T(ds).glob("*.csv")):
        inventory.append(dict(dataset=ds, **stat(p)))
for name, p in shared_paths.items():
    inventory.append(dict(dataset="shared", **stat(p)))

manifest = dict(created=datetime.datetime.now().isoformat(timespec="seconds"),
                roots={k: str(v) for k, v in ROOTS.items()},
                gate_ok=bool(GATE_OK), missing=missing, empty=empty,
                environment=env, inventory=inventory)
Path(FINAL / "final_manifest.json").write_text(json.dumps(manifest, indent=1))
print("final files:", sorted(p.name for p in FINAL.glob('*')))


final_core_tables_mvtec.json: 43 KB
final_core_tables_proditec.json: 37 KB
final files: ['final_core_tables_mvtec.json', 'final_core_tables_proditec.json', 'final_manifest.json', 'final_parser_eval.json', 'final_query_suite_detail.json', 'final_user_study.json']


In [6]:
# ============================================================
# F5) Human report + sanity checks
# ============================================================
L = ["=" * 70, "FINAL EXPORT REPORT", "=" * 70,
     f"created: {manifest['created']}", f"gate_ok: {GATE_OK}"]
if missing: L += ["MISSING:"] + [f"  - {m}" for m in missing]
if empty:   L += ["EMPTY:"] + [f"  - {m}" for m in empty]

L.append(""); L.append("SANITY:")
for ds in ROOTS:
    d = read_csv(T(ds) / f"duplication_rate_{ds}.csv")
    if d is not None:
        row = d[d["method"] == "CFNOW"]
        if len(row):
            v = float(row["duplicate_rate_pct"].iloc[0])
            L.append(f"  [{ds}] CFNOW dup rate = {v:.1f}% "
                     f"{'OK' if v >= 99 else '!!'}")
    b = read_csv(T(ds) / f"baseline_vs_generator_confirmed_{ds}.csv")
    L.append(f"  [{ds}] confirmed comparison rows = "
             f"{len(b) if b is not None else 'MISSING'}")
    e = read_csv(T(ds) / f"end_to_end_latency_{ds}.csv")
    if e is not None:
        L.append(f"  [{ds}] e2e latency range = "
                 f"{e['end_to_end_s'].min():.1f}-{e['end_to_end_s'].max():.1f}s, "
                 f"speedup {e['speedup_vs_retraining'].min():.0f}-"
                 f"{e['speedup_vs_retraining'].max():.0f}x")
pv = read_csv(shared_paths.get("parser_eval_v2_summary.csv"))      if "parser_eval_v2_summary.csv" in shared_paths else None
if pv is not None:
    for _, r in pv[pv["parser"] == "v2"].iterrows():
        if "SEMANTIC" in str(r["metric"]) or "EXACT full" in str(r["metric"]):
            L.append(f"  parser v2 | {r['metric']}: {r['value']:.3f}")
L.append("=" * 70)
report = "\n".join(L)
Path(FINAL / "final_report.txt").write_text(report)
print(report)
print()
print("UPLOAD THESE (<=7 files) from final_results/:")
for p in sorted(FINAL.glob("*")):
    print(f"  {p.name}  ({p.stat().st_size/1024:.0f} KB)")


FINAL EXPORT REPORT
created: 2026-07-26T22:37:54
gate_ok: True

SANITY:
  [mvtec] CFNOW dup rate = 100.0% OK
  [mvtec] confirmed comparison rows = 3
  [mvtec] e2e latency range = 141.4-241.2s, speedup 0-0x
  [proditec] CFNOW dup rate = 100.0% OK
  [proditec] confirmed comparison rows = 3
  [proditec] e2e latency range = 141.4-252.8s, speedup 0-0x
  parser v2 | EXACT full-query accuracy (all): 0.600
  parser v2 | SEMANTIC full-query accuracy (all): 0.625
  parser v2 | SEMANTIC on relative subset: 0.000

UPLOAD THESE (<=7 files) from final_results/:
  final_core_tables_mvtec.json  (43 KB)
  final_core_tables_proditec.json  (37 KB)
  final_manifest.json  (11 KB)
  final_parser_eval.json  (72 KB)
  final_query_suite_detail.json  (85 KB)
  final_report.txt  (1 KB)
  final_user_study.json  (26 KB)
